In [0]:
from pyspark.sql.functions import min, max, sum, avg

# Leitura da tabela de transações
df = spark.read.table("samples.bakehouse.sales_transactions")

# 1. Contagem total de registros na tabela/DataFrame
total_count = df.count()

# 2. Agregações simples sobre todo o dataset
df_agg = df.select(
    min("quantity").alias("min_quantity"),
    max("quantity").alias("max_quantity"),
    sum("quantity").alias("total_quantity"),
    avg("quantity").alias("avg_quantity")
)

display(df_agg)

In [0]:
%sql
SELECT 
    count(*) AS total_count,
    min(quantity) AS min_quantity,
    max(quantity) AS max_quantity,
    sum(quantity) AS total_quantity,
    avg(quantity) AS avg_quantity
FROM samples.bakehouse.sales_transactions;

In [0]:
from pyspark.sql.functions import count, avg

# Agrupando por forma de pagamento e calculando a contagem e média de quantidade
df_grouped = df.groupBy("paymentMethod").agg(
    count("*").alias("total_transactions"),
    avg("quantity").alias("avg_quantity")
)

display(df_grouped)

In [0]:
%sql
SELECT 
    paymentMethod,
    count(*) AS total_transactions,
    avg(quantity) AS avg_quantity
FROM samples.bakehouse.sales_transactions
GROUP BY paymentMethod;

In [0]:
from pyspark.sql.functions import count

# Agrupando e aplicando filtro sobre o resultado da agregação
df_having = df.groupBy("paymentMethod") \
              .agg(count("*").alias("count")) \
              .filter("count > 1100")

display(df_having)

In [0]:
%sql
SELECT 
    paymentMethod, 
    count(*) AS count
FROM samples.bakehouse.sales_transactions
GROUP BY paymentMethod
HAVING count(*) > 1100;

In [0]:
from pyspark.sql.functions import collect_set, collect_list

# Coletando os produtos comprados para cada método de pagamento
df_collections = df.groupBy("paymentMethod").agg(
    collect_set("product").alias("unique_products_array"),
    collect_list("product").alias("all_products_array")
)

display(df_collections)

In [0]:
%sql
SELECT 
    paymentMethod,
    collect_set(product) AS unique_products_array,
    collect_list(product) AS all_products_array
FROM samples.bakehouse.sales_transactions
GROUP BY paymentMethod;

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

# 1. Definindo a especificação da janela (WindowSpec)
# Agrupa por produto e ordena a quantidade de forma decrescente
windowSpec = Window.partitionBy("product").orderBy(col("quantity").desc())

# 2. Aplicando a função row_number() sobre a janela
df_window = df.withColumn("row_num", row_number().over(windowSpec))

# 3. Filtrando para obter o cliente que comprou a maior quantidade por produto (row_num == 1)
df_top_customer = df_window.select("product", "customerID", "quantity", "row_num") \
                           .filter(col("row_num") == 1)

display(df_top_customer)

In [0]:
%sql
WITH ranked_sales AS (
    SELECT 
        product,
        customerID,
        quantity,
        row_number() OVER (PARTITION BY product ORDER BY quantity DESC) AS row_num
    FROM samples.bakehouse.sales_transactions
)
SELECT * FROM ranked_sales WHERE row_num = 1;

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, col

# Especificação da Janela por produto ordenada pela quantidade decrescente
windowSpec = Window.partitionBy("product").orderBy(col("quantity").desc())

# Aplicando as três funções de ranking simultaneamente para comparar o comportamento
df_rankings = df.select("product", "customerID", "quantity") \
                .withColumn("row_num", row_number().over(windowSpec)) \
                .withColumn("rank", rank().over(windowSpec)) \
                .withColumn("dense_rank", dense_rank().over(windowSpec))

display(df_rankings)

In [0]:
%sql
SELECT 
    product,
    customerID,
    quantity,
    row_number() OVER (PARTITION BY product ORDER BY quantity DESC) AS row_num,
    rank() OVER (PARTITION BY product ORDER BY quantity DESC) AS rank,
    dense_rank() OVER (PARTITION BY product ORDER BY quantity DESC) AS dense_rank
FROM samples.bakehouse.sales_transactions;